In [2]:
import pandas as pd

# Set display options for better readability
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Load the CSV file
df = pd.read_csv('../data/train/train.csv')

# Display the first few rows and information about the DataFrame
print("Initial Data:")
print(df.head())
print("\nDataFrame Info:")
print(df.info())

# --- Data Preparation ---

# Convert '영업일자' to datetime objects
df['영업일자'] = pd.to_datetime(df['영업일자'])

# Add '요일' column (convert English day names to Korean)
day_mapping = {
    'Monday': '월요일',
    'Tuesday': '화요일',
    'Wednesday': '수요일',
    'Thursday': '목요일',
    'Friday': '금요일',
    'Saturday': '토요일',
    'Sunday': '일요일'
}
df['요일'] = df['영업일자'].dt.day_name().map(day_mapping)

# Split '영업장명_메뉴명' into '영업장명' and '메뉴명'
df[['영업장명', '메뉴명']] = df['영업장명_메뉴명'].str.split('_', n=1, expand=True)

# Drop the original '영업장명_메뉴명' column as it's no longer needed
df.drop('영업장명_메뉴명', axis=1, inplace=True)

# Reorder columns for better readability
df = df[['영업일자', '요일', '영업장명', '메뉴명', '매출수량']]

# Display the prepared data
print("\nPrepared Data:")
print(df.head())

# --- Analysis of Negative Sales ---

# Filter for rows with negative sales quantities
negative_sales_df = df[df['매출수량'] < 0].copy()

# Check if there are any negative sales records
if negative_sales_df.empty:
    print("\n'매출수량'이 음수인 데이터는 존재하지 않습니다. 따라서 분석을 진행할 수 없습니다.")
else:
    print("\n'매출수량'이 음수인 데이터:")
    print(negative_sales_df)

    # Initialize a list to store analysis results
    analysis_results = []

    # Iterate through each negative sales record
    for index, row in negative_sales_df.iterrows():
        target_date = row['영업일자']
        target_weekday = row['요일']
        target_store = row['영업장명']
        target_menu = row['메뉴명']

        # Define the time window for analysis (2 weeks before and 2 weeks after)
        # Note: TimeDelta is used for date arithmetic
        start_date = target_date - pd.Timedelta(weeks=2)
        end_date = target_date + pd.Timedelta(weeks=2)

        # Filter the original DataFrame for the same store, menu, and weekday within the time window
        surrounding_data = df[
            (df['영업장명'] == target_store) &
            (df['메뉴명'] == target_menu) &
            (df['요일'] == target_weekday) &
            (df['영업일자'] >= start_date) &
            (df['영업일자'] <= end_date) &
            (df['영업일자'] != target_date)
        ]

        if not surrounding_data.empty:
            avg_sales = surrounding_data['매출수량'].mean()
            median_sales = surrounding_data['매출수량'].median()

            analysis_results.append({
                '음수_매출_발생_일자': target_date.strftime('%Y-%m-%d'),
                '음수_매출_영업장': target_store,
                '음수_매출_메뉴': target_menu,
                '음수_매출_수량': row['매출수량'],
                '주변_요일_평균_매출': avg_sales,
                '주변_요일_중앙값_매출': median_sales,
                '주변_데이터': surrounding_data[['영업일자', '매출수량']].to_dict('records')
            })
        else:
            analysis_results.append({
                '음수_매출_발생_일자': target_date.strftime('%Y-%m-%d'),
                '음수_매출_영업장': target_store,
                '음수_매출_메뉴': target_menu,
                '음수_매출_수량': row['매출수량'],
                '주변_요일_평균_매출': '데이터 없음',
                '주변_요일_중앙값_매출': '데이터 없음',
                '주변_데이터': '데이터 없음'
            })

    # Print the final analysis results
    for result in analysis_results:
        print("\n--- 분석 결과 ---")
        print(f"발생 일자: {result['음수_매출_발생_일자']}")
        print(f"영업장: {result['음수_매출_영업장']}")
        print(f"메뉴: {result['음수_매출_메뉴']}")
        print(f"음수 매출 수량: {result['음수_매출_수량']}")
        print(f"주변 동요일 평균 매출: {result['주변_요일_평균_매출']:.2f}" if isinstance(result['주변_요일_평균_매출'], float) else f"주변 동요일 평균 매출: {result['주변_요일_평균_매출']}")
        print(f"주변 동요일 중앙값 매출: {result['주변_요일_중앙값_매출']:.2f}" if isinstance(result['주변_요일_중앙값_매출'], float) else f"주변 동요일 중앙값 매출: {result['주변_요일_중앙값_매출']}")
        print("--- 주변 데이터 ---")
        if result['주변_데이터'] == '데이터 없음':
            print("주변 분석 데이터가 없습니다.")
        else:
            for d in result['주변_데이터']:
                print(f"  - {d['영업일자'].strftime('%Y-%m-%d')}: {d['매출수량']}")

    # --- Recommendation ---
    print("\n--- 매출이 음수인 데이터 처리 방안 제안 ---")
    print("1. '매출수량'이 음수인 경우는 일반적으로 데이터 입력 오류로 간주할 수 있습니다.")
    print("2. 해당 데이터의 주변 동요일 매출 경향을 분석한 결과, 매출이 존재하는 경우가 많습니다.")
    print("3. 따라서, 음수 매출을 해당 영업장 및 메뉴의 주변 동요일 '평균' 또는 '중앙값' 매출로 대체하는 것이 합리적입니다.")
    print("   - '중앙값'은 이상치(Outlier)의 영향을 덜 받으므로, 더 안정적인 대안이 될 수 있습니다.")
    print("   - 예를 들어, 2023-03-22의 경우, 같은 요일(수요일)의 주변 매출 평균값인 4.70(중앙값 2.50)으로 대체하는 것을 고려해볼 수 있습니다.")
    print("4. 만약 음수 매출이 반복적으로 발생한다면, 데이터 입력 시스템 또는 영업 프로세스를 점검하여 원인을 파악하는 것이 중요합니다.")

Initial Data:
         영업일자            영업장명_메뉴명  매출수량
0  2023-01-01  느티나무 셀프BBQ_1인 수저세트     0
1  2023-01-02  느티나무 셀프BBQ_1인 수저세트     0
2  2023-01-03  느티나무 셀프BBQ_1인 수저세트     0
3  2023-01-04  느티나무 셀프BBQ_1인 수저세트     0
4  2023-01-05  느티나무 셀프BBQ_1인 수저세트     0

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102676 entries, 0 to 102675
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   영업일자      102676 non-null  object
 1   영업장명_메뉴명  102676 non-null  object
 2   매출수량      102676 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.4+ MB
None

Prepared Data:
        영업일자   요일        영업장명      메뉴명  매출수량
0 2023-01-01  일요일  느티나무 셀프BBQ  1인 수저세트     0
1 2023-01-02  월요일  느티나무 셀프BBQ  1인 수저세트     0
2 2023-01-03  화요일  느티나무 셀프BBQ  1인 수저세트     0
3 2023-01-04  수요일  느티나무 셀프BBQ  1인 수저세트     0
4 2023-01-05  목요일  느티나무 셀프BBQ  1인 수저세트     0

'매출수량'이 음수인 데이터:
            영업일자   요일        영업장명            메뉴명  매출수량
1837  2023

In [4]:
# Import necessary libraries
import pandas as pd

# Set display options for better readability
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Provided list of holidays
holidays_list = [
    '2023-01-01', '2023-01-21', '2023-01-22', '2023-01-23', '2023-01-24',
    '2023-03-01', '2023-05-01', '2023-05-05', '2023-05-27', '2023-06-06',
    '2023-08-15', '2023-09-28', '2023-09-29', '2023-09-30', '2023-10-02',
    '2023-10-03', '2023-10-09', '2023-12-25',
    '2024-01-01', '2024-02-09', '2024-02-10', '2024-02-11', '2024-02-12',
    '2024-03-01', '2024-04-19', '2024-05-01', '2024-05-05', '2024-05-06',
    '2024-05-15', '2024-06-06', '2024-08-15', '2024-09-16', '2024-09-17',
    '2024-09-18', '2024-10-01', '2024-10-03', '2024-10-09', '2024-12-25',
    '2025-01-01', '2025-01-28', '2025-01-29', '2025-01-30', '2025-03-01',
    '2025-03-03', '2025-05-01', '2025-05-05', '2025-05-06', '2025-06-06',
    '2025-08-15'
]
holidays_set = set(pd.to_datetime(holidays_list))

# Load the CSV file
df = pd.read_csv('../data/train/train.csv')

# Display the first few rows and information about the DataFrame
print("Initial Data:")
print(df.head())
print("\nDataFrame Info:")
print(df.info())

# --- Data Preparation ---

# Convert '영업일자' to datetime objects
df['영업일자'] = pd.to_datetime(df['영업일자'])

# Add '요일' column (convert English day names to Korean)
day_mapping = {
    'Monday': '월요일',
    'Tuesday': '화요일',
    'Wednesday': '수요일',
    'Thursday': '목요일',
    'Friday': '금요일',
    'Saturday': '토요일',
    'Sunday': '일요일'
}
df['요일'] = df['영업일자'].dt.day_name().map(day_mapping)

# Add '공휴일' column based on the provided list
df['공휴일'] = df['영업일자'].isin(holidays_set)

# Add '다음날공휴일' column
df['다음날공휴일'] = (df['영업일자'] + pd.Timedelta(days=1)).isin(holidays_set)

# Split '영업장명_메뉴명' into '영업장명' and '메뉴명'
df[['영업장명', '메뉴명']] = df['영업장명_메뉴명'].str.split('_', n=1, expand=True)

# Drop the original '영업장명_메뉴명' column as it's no longer needed
df.drop('영업장명_메뉴명', axis=1, inplace=True)

# Reorder columns for better readability
df = df[['영업일자', '요일', '공휴일', '다음날공휴일', '영업장명', '메뉴명', '매출수량']]

# Display the prepared data
print("\nPrepared Data:")
print(df.head())

# --- Analysis of Negative Sales ---

# Filter for rows with negative sales quantities
negative_sales_df = df[df['매출수량'] < 0].copy()

# Check if there are any negative sales records
if negative_sales_df.empty:
    print("\n'매출수량'이 음수인 데이터는 존재하지 않습니다. 따라서 분석을 진행할 수 없습니다.")
else:
    print("\n'매출수량'이 음수인 데이터:")
    print(negative_sales_df)

    # Initialize a list to store analysis results
    analysis_results = []

    # Iterate through each negative sales record
    for index, row in negative_sales_df.iterrows():
        target_date = row['영업일자']
        target_weekday = row['요일']
        target_store = row['영업장명']
        target_menu = row['메뉴명']

        # Define the time window for analysis (2 weeks before and 2 weeks after)
        start_date = target_date - pd.Timedelta(weeks=2)
        end_date = target_date + pd.Timedelta(weeks=2)

        # Filter the original DataFrame for the same store, menu, and weekday within the time window
        surrounding_data = df[
            (df['영업장명'] == target_store) &
            (df['메뉴명'] == target_menu) &
            (df['요일'] == target_weekday) &
            (df['영업일자'] >= start_date) &
            (df['영업일자'] <= end_date) &
            (df['영업일자'] != target_date)
        ]

        if not surrounding_data.empty:
            avg_sales = surrounding_data['매출수량'].mean()
            median_sales = surrounding_data['매출수량'].median()

            analysis_results.append({
                '음수_매출_발생_일자': target_date.strftime('%Y-%m-%d'),
                '음수_매출_영업장': target_store,
                '음수_매출_메뉴': target_menu,
                '음수_매출_수량': row['매출수량'],
                '주변_요일_평균_매출': avg_sales,
                '주변_요일_중앙값_매출': median_sales,
                '주변_데이터': surrounding_data[['영업일자', '요일', '공휴일', '다음날공휴일', '매출수량']].to_dict('records')
            })
        else:
            analysis_results.append({
                '음수_매출_발생_일자': target_date.strftime('%Y-%m-%d'),
                '음수_매출_영업장': target_store,
                '음수_매출_메뉴': target_menu,
                '음수_매출_수량': row['매출수량'],
                '주변_요일_평균_매출': '데이터 없음',
                '주변_요일_중앙값_매출': '데이터 없음',
                '주변_데이터': '데이터 없음'
            })

    # Print the final analysis results
    for result in analysis_results:
        print("\n--- 분석 결과 ---")
        print(f"발생 일자: {result['음수_매출_발생_일자']}")
        print(f"영업장: {result['음수_매출_영업장']}")
        print(f"메뉴: {result['음수_매출_메뉴']}")
        print(f"음수 매출 수량: {result['음수_매출_수량']}")
        print(f"주변 동요일 평균 매출: {result['주변_요일_평균_매출']:.2f}" if isinstance(result['주변_요일_평균_매출'], float) else f"주변 동요일 평균 매출: {result['주변_요일_평균_매출']}")
        print(f"주변 동요일 중앙값 매출: {result['주변_요일_중앙값_매출']:.2f}" if isinstance(result['주변_요일_중앙값_매출'], float) else f"주변 동요일 중앙값 매출: {result['주변_요일_중앙값_매출']}")
        print("--- 주변 데이터 ---")
        if result['주변_데이터'] == '데이터 없음':
            print("주변 분석 데이터가 없습니다.")
        else:
            for d in result['주변_데이터']:
                print(f"  - {d['영업일자'].strftime('%Y-%m-%d')}: {d['요일']}, 공휴일: {d['공휴일']}, 다음날 공휴일: {d['다음날공휴일']}, 매출수량: {d['매출수량']}")

    # --- Recommendation ---
    print("\n--- 매출이 음수인 데이터 처리 방안 제안 ---")
    print("1. '매출수량'이 음수인 경우는 일반적으로 데이터 입력 오류로 간주할 수 있습니다.")
    print("2. 해당 데이터의 주변 동요일 매출 경향을 분석한 결과, 매출이 존재하는 경우가 많습니다.")
    print("3. 따라서, 음수 매출을 해당 영업장 및 메뉴의 주변 동요일 '평균' 또는 '중앙값' 매출로 대체하는 것이 합리적입니다.")
    print("   - '중앙값'은 이상치(Outlier)의 영향을 덜 받으므로, 더 안정적인 대안이 될 수 있습니다.")
    print("4. 만약 음수 매출이 반복적으로 발생한다면, 데이터 입력 시스템 또는 영업 프로세스를 점검하여 원인을 파악하는 것이 중요합니다.")

Initial Data:
         영업일자            영업장명_메뉴명  매출수량
0  2023-01-01  느티나무 셀프BBQ_1인 수저세트     0
1  2023-01-02  느티나무 셀프BBQ_1인 수저세트     0
2  2023-01-03  느티나무 셀프BBQ_1인 수저세트     0
3  2023-01-04  느티나무 셀프BBQ_1인 수저세트     0
4  2023-01-05  느티나무 셀프BBQ_1인 수저세트     0

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102676 entries, 0 to 102675
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   영업일자      102676 non-null  object
 1   영업장명_메뉴명  102676 non-null  object
 2   매출수량      102676 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.4+ MB
None

Prepared Data:
        영업일자   요일    공휴일  다음날공휴일        영업장명      메뉴명  매출수량
0 2023-01-01  일요일   True   False  느티나무 셀프BBQ  1인 수저세트     0
1 2023-01-02  월요일  False   False  느티나무 셀프BBQ  1인 수저세트     0
2 2023-01-03  화요일  False   False  느티나무 셀프BBQ  1인 수저세트     0
3 2023-01-04  수요일  False   False  느티나무 셀프BBQ  1인 수저세트     0
4 2023-01-05  목요일  False   False  느티나무 셀프BBQ  1인 수저세트

In [6]:
# Import necessary libraries
import pandas as pd

# Set display options for better readability
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Provided holiday list as a string
holiday_list_text = """
[2023]
1/1(일)
1/21(토)
1/22(일)
1/23(월)
1/24(화)
3/1(수)
5/1(월)
5/5(금)
5/27(토)
6/6(화)
8/15(화)
9/28(목)
9/29(금)
9/30(토)
10/2(월)
10/3(화)
10/9(월)
12/25(월)

[2024]
1/1(월)
2/9(금)
2/10(일)
2/11(월)
2/12(화)
3/1(금)
4/19(수)
5/1(수)
5/5(일)
5/6(월)
5/15(수)
6/6(목)
8/15(목)
9/16(월)
9/17(화)
9/18(수)
10/1(화)
10/3(목)
10/9(수)
12/25(수)

[2025]
1/1(수)
1/28(화)
1/29(수)
1/30(목)
3/1(토)
3/3(월)
5/1(목)
5/5(월)
5/6(화)
6/6(금)
8/15(금)
"""

# Process the holiday list into a set of datetime objects for efficient lookup
holiday_dates = set()
current_year = None
for line in holiday_list_text.strip().split('\n'):
    line = line.strip()
    if not line:
        continue
    if line.startswith('[') and line.endswith(']'):
        current_year = int(line[1:-1])
    elif current_year:
        try:
            month_day_part = line.split('(')[0]
            month, day = map(int, month_day_part.split('/'))
            holiday_date_str = f'{current_year}-{month:02d}-{day:02d}'
            holiday_dates.add(pd.to_datetime(holiday_date_str))
        except (ValueError, IndexError):
            continue

# Load the CSV file
df = pd.read_csv('../data/train/train.csv')

# Display the first few rows and information about the DataFrame
print("Initial Data:")
print(df.head())
print("\nDataFrame Info:")
print(df.info())

# --- Data Preparation ---

# Convert '영업일자' to datetime objects
df['영업일자'] = pd.to_datetime(df['영업일자'])

# Add '요일' column (convert English day names to Korean)
day_mapping = {
    'Monday': '월요일',
    'Tuesday': '화요일',
    'Wednesday': '수요일',
    'Thursday': '목요일',
    'Friday': '금요일',
    'Saturday': '토요일',
    'Sunday': '일요일'
}
df['요일'] = df['영업일자'].dt.day_name().map(day_mapping)

# Add '쉬는날' column (Saturday, Sunday, or a holiday)
weekend_dates = set(df[df['영업일자'].dt.dayofweek >= 5]['영업일자'])
day_off_dates = holiday_dates.union(weekend_dates)
df['쉬는날'] = df['영업일자'].isin(day_off_dates)

# Add '다음날쉬는날' column
df['다음날쉬는날'] = (df['영업일자'] + pd.Timedelta(days=1)).isin(day_off_dates)

# Split '영업장명_메뉴명' into '영업장명' and '메뉴명'
df[['영업장명', '메뉴명']] = df['영업장명_메뉴명'].str.split('_', n=1, expand=True)

# Drop the original '영업장명_메뉴명' column as it's no longer needed
df.drop('영업장명_메뉴명', axis=1, inplace=True)

# Reorder columns for better readability
df = df[['영업일자', '요일', '쉬는날', '다음날쉬는날', '영업장명', '메뉴명', '매출수량']]

# Display the prepared data
print("\nPrepared Data:")
print(df.head())

# --- Analysis of Negative Sales ---

# Filter for rows with negative sales quantities
negative_sales_df = df[df['매출수량'] < 0].copy()

# Check if there are any negative sales records
if negative_sales_df.empty:
    print("\n'매출수량'이 음수인 데이터는 존재하지 않습니다. 따라서 분석을 진행할 수 없습니다.")
else:
    print("\n'매출수량'이 음수인 데이터:")
    print(negative_sales_df)

    # Initialize a list to store analysis results
    analysis_results = []

    # Iterate through each negative sales record
    for index, row in negative_sales_df.iterrows():
        target_date = row['영업일자']
        target_weekday = row['요일']
        target_day_off = row['쉬는날']
        target_next_day_off = row['다음날쉬는날']
        target_store = row['영업장명']
        target_menu = row['메뉴명']

        # Define the time window for analysis (2 weeks before and 2 weeks after)
        start_date = target_date - pd.Timedelta(weeks=2)
        end_date = target_date + pd.Timedelta(weeks=2)

        # Filter the original DataFrame for the same store, menu, and weekday within the time window
        surrounding_data = df[
            (df['영업장명'] == target_store) &
            (df['메뉴명'] == target_menu) &
            (df['요일'] == target_weekday) &
            (df['영업일자'] >= start_date) &
            (df['영업일자'] <= end_date) &
            (df['영업일자'] != target_date)
        ]

        if not surrounding_data.empty:
            avg_sales = surrounding_data['매출수량'].mean()
            median_sales = surrounding_data['매출수량'].median()

            analysis_results.append({
                '음수_매출_발생_일자': target_date.strftime('%Y-%m-%d'),
                '음수_매출_요일': target_weekday,
                '음수_매출_쉬는날': target_day_off,
                '음수_매출_다음날쉬는날': target_next_day_off,
                '음수_매출_영업장': target_store,
                '음수_매출_메뉴': target_menu,
                '음수_매출_수량': row['매출수량'],
                '주변_요일_평균_매출': avg_sales,
                '주변_요일_중앙값_매출': median_sales,
                '주변_데이터': surrounding_data[['영업일자', '요일', '쉬는날', '다음날쉬는날', '매출수량']].to_dict('records')
            })
        else:
            analysis_results.append({
                '음수_매출_발생_일자': target_date.strftime('%Y-%m-%d'),
                '음수_매출_요일': target_weekday,
                '음수_매출_쉬는날': target_day_off,
                '음수_매출_다음날쉬는날': target_next_day_off,
                '음수_매출_영업장': target_store,
                '음수_매출_메뉴': target_menu,
                '음수_매출_수량': row['매출수량'],
                '주변_요일_평균_매출': '데이터 없음',
                '주변_요일_중앙값_매출': '데이터 없음',
                '주변_데이터': '데이터 없음'
            })

    # Print the final analysis results
    for result in analysis_results:
        print("\n--- 분석 결과 ---")
        print(f"발생 일자: {result['음수_매출_발생_일자']}")
        print(f"발생 요일: {result['음수_매출_요일']}")
        print(f"쉬는날 여부: {result['음수_매출_쉬는날']}")
        print(f"다음날 쉬는날 여부: {result['음수_매출_다음날쉬는날']}")
        print(f"영업장: {result['음수_매출_영업장']}")
        print(f"메뉴: {result['음수_매출_메뉴']}")
        print(f"음수 매출 수량: {result['음수_매출_수량']}")
        print(f"주변 동요일 평균 매출: {result['주변_요일_평균_매출']:.2f}" if isinstance(result['주변_요일_평균_매출'], float) else f"주변 동요일 평균 매출: {result['주변_요일_평균_매출']}")
        print(f"주변 동요일 중앙값 매출: {result['주변_요일_중앙값_매출']:.2f}" if isinstance(result['주변_요일_중앙값_매출'], float) else f"주변 동요일 중앙값 매출: {result['주변_요일_중앙값_매출']}")
        print("--- 주변 데이터 ---")
        if result['주변_데이터'] == '데이터 없음':
            print("주변 분석 데이터가 없습니다.")
        else:
            for d in result['주변_데이터']:
                print(f"  - 날짜: {d['영업일자'].strftime('%Y-%m-%d')}, 요일: {d['요일']}, 쉬는날: {d['쉬는날']}, 다음날쉬는날: {d['다음날쉬는날']}, 매출수량: {d['매출수량']}")

    # --- Recommendation ---
    print("\n--- 매출이 음수인 데이터 처리 방안 제안 ---")
    print("1. '매출수량'이 음수인 경우는 일반적으로 데이터 입력 오류로 간주할 수 있습니다.")
    print("2. 해당 데이터의 주변 동요일 매출 경향을 분석한 결과, 매출이 존재하는 경우가 많습니다.")
    print("3. 따라서, 음수 매출을 해당 영업장 및 메뉴의 주변 동요일 '평균' 또는 '중앙값' 매출로 대체하는 것이 합리적입니다.")
    print("   - '중앙값'은 이상치(Outlier)의 영향을 덜 받으므로, 더 안정적인 대안이 될 수 있습니다.")
    print("4. 만약 음수 매출이 반복적으로 발생한다면, 데이터 입력 시스템 또는 영업 프로세스를 점검하여 원인을 파악하는 것이 중요합니다.")

Initial Data:
         영업일자            영업장명_메뉴명  매출수량
0  2023-01-01  느티나무 셀프BBQ_1인 수저세트     0
1  2023-01-02  느티나무 셀프BBQ_1인 수저세트     0
2  2023-01-03  느티나무 셀프BBQ_1인 수저세트     0
3  2023-01-04  느티나무 셀프BBQ_1인 수저세트     0
4  2023-01-05  느티나무 셀프BBQ_1인 수저세트     0

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102676 entries, 0 to 102675
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   영업일자      102676 non-null  object
 1   영업장명_메뉴명  102676 non-null  object
 2   매출수량      102676 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 2.4+ MB
None

Prepared Data:
        영업일자   요일    쉬는날  다음날쉬는날        영업장명      메뉴명  매출수량
0 2023-01-01  일요일   True   False  느티나무 셀프BBQ  1인 수저세트     0
1 2023-01-02  월요일  False   False  느티나무 셀프BBQ  1인 수저세트     0
2 2023-01-03  화요일  False   False  느티나무 셀프BBQ  1인 수저세트     0
3 2023-01-04  수요일  False   False  느티나무 셀프BBQ  1인 수저세트     0
4 2023-01-05  목요일  False   False  느티나무 셀프BBQ  1인 수저세트